### Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

### Drug sensitivity info extraction

In [2]:
batch_sen = pd.read_csv("../data/rawdata/uhn_breast/batch_metrics.csv")

# batch name contains PACLITAXEL but not "-"
pax = batch_sen[batch_sen["batch.name"].str.contains("PACLITAXEL") & ~batch_sen["batch.name"].str.contains("-") & ~batch_sen["batch.name"].str.contains("RES")].reset_index(drop=True)

pax["model.id"] = pax["batch.name"].str.split(".").str[0]
pax["drug.id"] = pax["batch.name"].str.split(".").str[1]

mask = pax["model.id"].str.match(r'^\d', na=False)
pax.loc[mask, "model.id"] = "S" + pax.loc[mask, "model.id"]

pax = pax[["model.id", "angle", "mRECIST", "TGI"]].set_index("model.id")
pax.to_csv("../data/procdata/4_paclitaxel/metadata.csv")

### RNAseq data

In [3]:
# read in the RNA-seq data
rna = pd.read_csv("../data/rawdata/uhn_breast/omics/rna_tpm_hugo.tsv", sep="	", index_col=0)

rna = rna.transpose()
print(f"Models in RNA-seq: {len(rna)}")
rna = np.log2(rna + 1)

# remove genes with no variance across all available RNA-seq samples
rna = rna.loc[:, rna.var() > 0]

# merge with paclitaxel response metadata
df_rna_pax = rna.merge(pax, left_index=True, right_index=True, how="inner")

# remove genes with no variance within the paclitaxel subset
response_cols = ["angle", "mRECIST", "TGI"]
X = df_rna_pax.drop(columns=response_cols)
y = df_rna_pax[response_cols]
X = X.loc[:, X.var() > 0]
df_rna_pax = X.merge(y, left_index=True, right_index=True)

print(f"Models in pax list: {len(pax)}")
print(f"Models in Final DF: {df_rna_pax.shape[0]}")
print(f"RNA genes after paclitaxel subset zero-variance filter: {X.shape[1]}")

df_rna_pax.to_csv("../data/procdata/4_paclitaxel/rna_logtpm.csv")
print(df_rna_pax.shape)


Models in RNA-seq: 95
Models in pax list: 80
Models in Final DF: 67
RNA genes after paclitaxel subset zero-variance filter: 49348


(67, 49351)


### CNV data

In [4]:
# read in the CNV data
cnv = pd.read_csv("../data/rawdata/uhn_breast/omics/cnv_hugo.tsv", sep="	", index_col=0)

# transpose to samples x genes
cnv = cnv.transpose()
print(f"Models in CNV before filtering: {len(cnv)}")

# remove genes with high missingness across samples
missing_rate = cnv.isna().mean(axis=0)
missing_rate.to_csv("../data/procdata/4_paclitaxel/cnv_gene_missing_rate.csv")

cnv = cnv.loc[:, missing_rate < 0.20]
print(f"CNV genes after missingness filter: {cnv.shape[1]}")

# impute remaining missing values using sample-wise median copy number
# this preserves sample-level ploidy/aneuploidy better than filling with 2
cnv = cnv.apply(lambda row: row.fillna(row.median()), axis=1)

# remove genes with no variance after imputation across all available CNV samples
cnv = cnv.loc[:, cnv.var() > 0]
print(f"CNV genes after global zero-variance filter: {cnv.shape[1]}")

# merge with paclitaxel response metadata
df_cnv_pax = cnv.merge(pax, left_index=True, right_index=True, how="inner")

# remove genes with no variance within the paclitaxel subset
response_cols = ["angle", "mRECIST", "TGI"]
X = df_cnv_pax.drop(columns=response_cols)
y = df_cnv_pax[response_cols]
X = X.loc[:, X.var() > 0]
df_cnv_pax = X.merge(y, left_index=True, right_index=True)

print(f"Models in pax list: {len(pax)}")
print(f"Models in Final DF: {df_cnv_pax.shape[0]}")
print(f"CNV genes after paclitaxel subset zero-variance filter: {X.shape[1]}")

# save processed CNV + response table
df_cnv_pax.to_csv("../data/procdata/4_paclitaxel/cnv.csv")
print(df_cnv_pax.shape)


Models in CNV before filtering: 94
CNV genes after missingness filter: 10004
CNV genes after global zero-variance filter: 9961
Models in pax list: 80
Models in Final DF: 62
CNV genes after paclitaxel subset zero-variance filter: 9956


(62, 9959)


### Mutation data

In [5]:
# read in the Mutation data
mutation = pd.read_csv("../data/rawdata/uhn_breast/omics/mutation_hugo.tsv", sep="	", index_col=0)

mutation = mutation.transpose()

# Define non-functional terms
non_functional_terms = {"Silent", 
                        "Intron",
                        "3'UTR",
                        "5'UTR",
                        "3'Flank",
                        "5'Flank",
                        "RNA",
                        "IGR",
                        "Targeted_Region"}

def binarize_mutation(val):
    if pd.isna(val) or val == "":
        return 0
    
    # Split by comma in case of multiple annotations (e.g., "Intron,Missense")
    parts = {p.strip() for p in str(val).split(',')}
    
    # If all parts are in the non_functional list, return 0
    if parts.issubset(non_functional_terms):
        return 0
    
    # Otherwise, there is at least one functional mutation
    return 1

# Apply the function (using .map for modern pandas)
mutation = mutation.map(binarize_mutation)

print(f"Models in Mutation: {len(mutation)}")

# remove genes with no variance across all available mutation samples
mutation = mutation.loc[:, mutation.var() > 0]

# merge with paclitaxel response metadata
df_mutation_pax = mutation.merge(pax, left_index=True, right_index=True, how="inner")

# remove genes with no variance within the paclitaxel subset
response_cols = ["angle", "mRECIST", "TGI"]
X = df_mutation_pax.drop(columns=response_cols)
y = df_mutation_pax[response_cols]
X = X.loc[:, X.var() > 0]
df_mutation_pax = X.merge(y, left_index=True, right_index=True)

print(f"Models in pax list: {len(pax)}")
print(f"Models in Final DF: {df_mutation_pax.shape[0]}")
print(f"Mutation genes after paclitaxel subset zero-variance filter: {X.shape[1]}")

df_mutation_pax.to_csv("../data/procdata/4_paclitaxel/mutation.csv")
print(df_mutation_pax.shape)


Models in Mutation: 91
Models in pax list: 80
Models in Final DF: 62
Mutation genes after paclitaxel subset zero-variance filter: 13642
(62, 13645)
